In [ ]:
#TODO: load data
#TODO: train val test split
#TODO: missing data 
#TODO: outliers
#TODO: correlation (transofrming)
#TODO: scaling

### Load data

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Find MAL folder no matter if the notebook is run from MAL/ or MAL/notebooks/
current_dir = Path.cwd()
MAL_DIR = current_dir if (current_dir / "data").exists() else current_dir.parent

DATA_DIR = MAL_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Later, when the data team gives us a final file, we only need to change this path.
DATASET_CANDIDATES = [
    PROCESSED_DATA_DIR / "focus_dataset.csv",
    DATA_DIR / "mock" / "new" / "data.csv",
    RAW_DATA_DIR / "room_conditions.csv",
]

dataset_path = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError("No dataset found. Check data/raw, data/interim, or data/processed.")

df = pd.read_csv(dataset_path)

# Small generic cleanup that is safe for most CSVs
df = df.drop(columns=[col for col in df.columns if col.lower().startswith("unnamed")], errors="ignore")
df.columns = df.columns.str.strip()
df = df.reset_index(drop=True)

print(f"Loaded dataset: {dataset_path}")
print(f"Shape: {df.shape}")

display(df.head())
display(df.dtypes)
display(df.isna().sum().rename("missing_values"))


### Train val test split

In [ ]:
from sklearn.model_selection import train_test_split

TARGET_COLUMN = "rating"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        "This dataset can be cleaned/preprocessed, but it cannot be split for supervised training yet."
    )

# Keep only numeric features for now.
# Later preprocessing steps can handle dates, categories, scaling, etc.
columns_to_ignore = [
    TARGET_COLUMN,
    "timePeriod",
    "session_id",
    "sent_at",
    "date",
    "Date",
    "id",
]

feature_columns = [
    column for column in df.columns
    if column not in columns_to_ignore and pd.api.types.is_numeric_dtype(df[column])
]

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

# Remove rows where the target is missing.
valid_target_rows = y.notna()
X = X[valid_target_rows]
y = y[valid_target_rows]

# For now rating is treated as classes.
# If we later test regression, we can still reuse the same split and change the model/evaluation.
y = y.astype(int)

# Stratify only works if every class has at least 2 rows.
class_counts = y.value_counts()
stratify_target = y if class_counts.min() >= 2 else None

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=stratify_target,
)

train_val_stratify = y_train_val if y_train_val.value_counts().min() >= 2 else None

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,  # gives roughly 15% validation, 70% train, 15% test overall
    random_state=42,
    stratify=train_val_stratify,
)

print("Feature columns:")
print(feature_columns)

print("\nSplit sizes:")
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Validation: {X_val.shape}, {y_val.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

print("\nTarget distribution:")
display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "validation": y_val.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).fillna(0))

### Missing data

In [ ]:
#For missing values we decided to keep the first version simple: if a row has a missing value, we remove that row. We expect the final dataset to have very few or no missing values, because the sensor/data preparation step should produce complete rows. Dropping rows is easier to explain and avoids guessing values with mean/median imputation. If we later see that many rows are removed, then we should reconsider and use imputation instead.
# TODO: change this

missing_before = df.isna().sum().sum()
rows_before = len(df)

df = df.dropna().reset_index(drop=True)

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Missing values before: {missing_before}")
print(f"Rows before: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows after: {rows_after}")

display(df.isna().sum().rename("missing_values_after"))


### Outliers